# GA4 Churn Analysis — Google Colab

Bu notebook GA4 BigQuery export verisinde **user_id bazlı** active / pre-churn / churn analizi çalıştırır.

> `user_id` boş olan kayıtlar analize dahil edilmez.


In [ ]:
%pip install -q --upgrade "ga4-churn-toolkit @ git+https://github.com/yasinsariyildizz/ga4-churn-toolkit.git@main"


## 1. Kaynak bilgilerini gir


In [ ]:
PROJECT_ID = "your-gcp-project" # @param {type:"string"}
DATASET_ID = "analytics_123456789" # @param {type:"string"}
TABLE_ID = "events_*" # @param {type:"string"}
OUTPUT_DATASET_ID = "ga4_churn" # @param {type:"string"}


## 2. Google hesabınla yetkilendir


In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Google Cloud yetkilendirmesi tamamlandı.")


## 3. Analizi başlat


In [ ]:
from ga4_churn import ChurnAnalysis
analysis = ChurnAnalysis(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    output_dataset_id=OUTPUT_DATASET_ID,
)


## 4. Maliyet kontrolü


In [ ]:
analysis.dry_run()


## 5. user_id bazlı temel tabloyu oluştur

Her `user_id` için alışveriş, oturum, gelir ve son alışverişten bu yana geçen gün bilgisini oluşturur. `user_id` boş kayıtlar dışarıda kalır.


In [ ]:
analysis.create_base_table()


## 6. Satın alma aralıklarını incele


In [ ]:
analysis.purchase_day_distribution()


## 7. Pre-churn ve churn sınırlarını seç

Örnek: `PRE_CHURN_THRESHOLD = 60`, `CHURN_THRESHOLD = 90` seçersen:

- Son alışverişten **0–60 gün** geçtiyse: `active_purchaser`
- **61–90 gün** geçtiyse: `pre_churn`
- **90 günden fazla** geçtiyse: `churned`

Pre-churn değeri churn değerinden küçük olmalıdır.


In [ ]:
PRE_CHURN_THRESHOLD = 60 # @param {type:"integer"}
CHURN_THRESHOLD = 90 # @param {type:"integer"}

churn_result = analysis.churn_analysis(
    PRE_CHURN_THRESHOLD,
    CHURN_THRESHOLD,
)


## 8. HTML dashboard'u görüntüle


In [ ]:
from pathlib import Path
from IPython.display import HTML, display
dashboard_path = Path(churn_result["dashboard_path"])
display(HTML(dashboard_path.read_text(encoding="utf-8")))


## 9. HTML dashboard'u indir


In [ ]:
from google.colab import files
files.download(churn_result["dashboard_path"])


## Önemli

Bu sürüm yalnızca `user_id` dolu kullanıcıları analiz eder. GA4'te `user_id` sadece login olan veya sizin tarafınızdan kimliği atanmış kullanıcılarda doluyorsa, sonuçlar tüm site kullanıcılarını değil bu tanımlı kullanıcı kitlesini temsil eder.
